# Session 1 - Train/Test Split + Baseline Linear Regression

Goal: build the first regression baseline by reusing the preprocessing code from Milestone 1, splitting the data, fitting `LinearRegression`, and checking a few predictions.

Today is about understanding the workflow. Model accuracy can be improved later.

## 1. Key Concepts

### 1.1. Reuse preprocessing

- `preprocessing.py` keeps data cleaning, encoding, and scaling in one reusable module.
- The notebook imports that module instead of rewriting preprocessing steps.
- Use notebooks for experiments. Use `.py` modules for reusable logic.

### 1.2. Train/test split

- `X` is the feature data. `y` is the target, here `SalePrice`.
- The model trains on the train set and is checked on the test set.
- `test_size=0.2` means 80% train and 20% test.
- `random_state=42` makes the split repeatable.
- Data leakage happens when test data affects training. A stricter flow should split first, fit preprocessing on train, then transform test.

### 1.3. Linear Regression baseline

- `LinearRegression` is a simple first model.
- `coef_` shows how each feature affects the prediction.
- A positive coefficient means the prediction tends to increase.
- A negative coefficient means the prediction tends to decrease.
- `intercept_` is the base value of the model. It is usually not a real house price.

### 1.4. Dummy baseline

- `DummyRegressor(strategy="mean")` always predicts the average target value.
- Linear Regression should do better than this dummy model.
- This notebook trains on `np.log1p(SalePrice)` and uses `np.expm1()` to convert predictions back to prices.

In [2]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Set Up Paths And Import Preprocessing

Add `my-project/src` to `sys.path` so this notebook can import `ml.preprocessing`.

In [3]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "model_training":
    PROJECT_DIR = PROJECT_DIR.parents[1]
elif PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / "my-project").exists():
    PROJECT_DIR = PROJECT_DIR / "my-project"

SRC_DIR = PROJECT_DIR / "src"
DATA_PATH = PROJECT_DIR / "data" / "raw" / "train.csv"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
PREPROCESSOR_PATH = ARTIFACTS_DIR / "preprocessor_baseline.pkl"

sys.path.insert(0, str(SRC_DIR))

from ml.preprocessing import preprocess_train

print(f"Project dir: {PROJECT_DIR}")
print(f"Data path: {DATA_PATH}")
print(f"Preprocessor artifact: {PREPROCESSOR_PATH}")

Project dir: c:\Users\ADMIN\OneDrive - The University of Technology\Documents\Intern AgilityIO\internship-hoaihuynh-training\my-project
Data path: c:\Users\ADMIN\OneDrive - The University of Technology\Documents\Intern AgilityIO\internship-hoaihuynh-training\my-project\data\raw\train.csv
Preprocessor artifact: c:\Users\ADMIN\OneDrive - The University of Technology\Documents\Intern AgilityIO\internship-hoaihuynh-training\my-project\artifacts\preprocessor_baseline.pkl


## 3. Load Raw Data And Split Target

The Kaggle `train.csv` file should be at `my-project/data/raw/train.csv`. Data files are local and are not committed to Git.

In [7]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing training data: {DATA_PATH}. "
        "Place Kaggle House Prices train.csv under my-project/data/raw/ first."
    )

df_raw = pd.read_csv(DATA_PATH)

if "SalePrice" not in df_raw.columns:
    raise ValueError("Expected target column `SalePrice` in train.csv")

y_price = df_raw["SalePrice"]
x_raw = df_raw.drop(columns=["SalePrice"])

print(f"Raw feature shape: {x_raw.shape}")
print(f"Target shape: {y_price.shape}")
display(df_raw.head())

Raw feature shape: (1460, 80)
Target shape: (1460,)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 4. Select Features For The First Baseline

Use a small feature set from `src/run_pipeline.py` so this session stays focused on the training workflow.

In [8]:
num_cols = ["LotFrontage", "MasVnrArea", "TotalBsmtSF"]
cat_cols = ["GarageType", "Alley"]
ord_cols = ["ExterQual"]

selected_cols = num_cols + cat_cols + ord_cols
missing_cols = [col for col in selected_cols if col not in x_raw.columns]

if missing_cols:
    raise ValueError(f"Missing selected columns in train.csv: {missing_cols}")

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Ordinal columns:", ord_cols)

Numeric columns: ['LotFrontage', 'MasVnrArea', 'TotalBsmtSF']
Categorical columns: ['GarageType', 'Alley']
Ordinal columns: ['ExterQual']


## 5. Preprocess Features And Transform Target

`preprocess_train()` cleans the data, creates features, fits the transformer, transforms features, and saves the preprocessor to `my-project/artifacts/preprocessor_baseline.pkl`.

`y_log = np.log1p(SalePrice)` makes the target easier for the model to learn.

**Leakage note:** This notebook preprocesses all `x_raw` before splitting, which is okay for this first learning exercise. Later, improve it by splitting raw data first, fitting preprocessing on train only, and transforming test after that.

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

x_processed = preprocess_train(
    df=x_raw,
    num_cols=num_cols,
    cat_cols=cat_cols,
    ord_cols=ord_cols,
    save_path=str(PREPROCESSOR_PATH),
)
y_log = np.log1p(y_price)

print(f"Processed X shape: {x_processed.shape}")
print(f"Log target shape: {y_log.shape}")
print(f"Saved preprocessor: {PREPROCESSOR_PATH.exists()}")

## 6. Train/Test Split

Use `test_size=0.2` for a 80/20 split and `random_state=42` for repeatable results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_processed,
    y_log,
    test_size=0.2,
    random_state=42,
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

## 7. Fit Linear Regression baseline

This is the first baseline model. It gives us a simple result to compare with later models.

`intercept_` is the model base value. In this notebook, do not read it as a real house price.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_log = linear_model.predict(X_test)

rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log = mean_absolute_error(y_test, y_pred_log)
r2 = r2_score(y_test, y_pred_log)

print("Linear Regression baseline")
print(f"Intercept: {linear_model.intercept_:.4f}")
print(f"RMSE on log target: {rmse_log:.4f}")
print(f"MAE on log target: {mae_log:.4f}")
print(f"R2: {r2:.4f}")

## 8. Read Model Coefficients

After one-hot encoding, there are more features than the original columns. Use the fitted preprocessor to get the final feature names.

In [ ]:
preprocessor = joblib.load(PREPROCESSOR_PATH)
feature_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": linear_model.coef_,
    }
)
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

display(coef_df.head(10))

## 9. Coefficient Notes

- A positive coefficient means the prediction tends to increase.
- A negative coefficient means the prediction tends to decrease.
- `num__TotalBsmtSF` should usually be positive because larger houses often cost more.
- One-hot features like `cat__GarageType_*` should be read carefully.

This baseline uses only a few features. It is for learning, not for final accuracy.

## 10. Predict Samples And Compare With Actual Values

In [ ]:
sample_size = 5
sample_actual_log = y_test.iloc[:sample_size].to_numpy()
sample_pred_log = y_pred_log[:sample_size]

prediction_samples = pd.DataFrame(
    {
        "actual_log": sample_actual_log,
        "predicted_log": sample_pred_log,
        "actual_price": np.expm1(sample_actual_log),
        "predicted_price": np.expm1(sample_pred_log),
    }
)
prediction_samples["error_price"] = (
    prediction_samples["predicted_price"] - prediction_samples["actual_price"]
)

display(prediction_samples)

## 11. Optional: Dummy Baseline

`DummyRegressor(strategy="mean")` always predicts the train target average. Linear Regression should perform better than this.

In [ ]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

dummy_pred_log = dummy_model.predict(X_test)
dummy_rmse_log = np.sqrt(mean_squared_error(y_test, dummy_pred_log))
dummy_mae_log = mean_absolute_error(y_test, dummy_pred_log)
dummy_r2 = r2_score(y_test, dummy_pred_log)

comparison = pd.DataFrame(
    [
        {
            "model": "DummyRegressor(mean)",
            "rmse_log": dummy_rmse_log,
            "mae_log": dummy_mae_log,
            "r2": dummy_r2,
        },
        {
            "model": "LinearRegression",
            "rmse_log": rmse_log,
            "mae_log": mae_log,
            "r2": r2,
        },
    ]
)

display(comparison)

## Session Summary

- Reused preprocessing instead of rewriting data cleaning logic.
- Created an 80/20 train/test split with `random_state=42`.
- Trained the first `LinearRegression` baseline.
- Checked coefficients and sample predictions.
- Compared Linear Regression with a dummy baseline.